# 02 — Fine-tune BART-base on SciTLDR

**Purpose:** fine-tune the original `facebook/bart-base` checkpoint on abstract → human TLDR pairs.

**Your responsibility:** train and save one reusable model checkpoint, with training logs and validation-loss monitoring. This notebook does not generate the five reranking candidates or calculate AlignScore/SummaC.

**Input:** `data/processed/train.jsonl` and `val.jsonl` from Notebook 01  
**Outputs:** `outputs/model/` and `outputs/train_log.json`

The training split has one reference per paper, so each paper contributes one supervised pair. The validation split may have multiple references; for checkpoint selection, this notebook uses the first reference consistently. All references remain preserved in the processed data for later multi-reference evaluation.


In [ ]:
from pathlib import Path
import json, math, os, inspect, time
import pandas as pd
import torch
import transformers
import yaml
from IPython.display import display
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
    Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback, set_seed
)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CFG = yaml.safe_load((ROOT / "configs" / "config.yaml").read_text())
D, T, SEED = CFG["data"], CFG["train"], CFG["seed"]
MODEL_DIR = ROOT / "outputs" / "model"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("Device:", "CUDA / " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## 1. Load processed training and validation data

Run Notebook 01 first. The test set is deliberately not loaded here.


In [ ]:
def read_split(name):
    path = ROOT / "data" / "processed" / f"{name}.jsonl"
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. Run 01_data_preparation.ipynb first.")
    return pd.read_json(path, orient="records", lines=True)

train_df = read_split("train")
val_df = read_split("val")

assert len(train_df) > 0 and len(val_df) > 0
assert train_df["human_tldrs"].map(len).eq(1).all(), "Training split should have exactly one TLDR per paper."

print(f"Training papers: {len(train_df)}")
print(f"Validation papers: {len(val_df)}")
print("Validation references per paper:")
display(val_df["human_tldrs"].map(len).value_counts().sort_index().rename("papers").to_frame())


## 2. Choose a safe batch size for the available GPU

The effective batch size remains the config value (16). The physical batch size is reduced on smaller GPUs and gradient accumulation makes up the difference. BART-base fine-tuning may still exceed 4 GB VRAM; use a Colab/Kaggle GPU if needed.


In [ ]:
set_seed(SEED)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    # Conservative starting choice; gradient checkpointing is also enabled below.
    per_device_bs = 4 if vram_gb >= 10 else (2 if vram_gb >= 6 else 1)
    use_fp16 = torch.cuda.get_device_capability(0)[0] < 8
    use_bf16 = torch.cuda.get_device_capability(0)[0] >= 8
else:
    gpu_name, vram_gb = "CPU", 0
    per_device_bs, use_fp16, use_bf16 = 1, False, False

effective_batch = int(T["effective_batch"])
assert effective_batch % per_device_bs == 0
grad_accum = effective_batch // per_device_bs

print(f"Device: {gpu_name} | VRAM: {vram_gb:.1f} GB")
print(f"Per-device batch: {per_device_bs} | gradient accumulation: {grad_accum}")
print(f"Effective batch size: {per_device_bs * grad_accum}")
print(f"fp16: {use_fp16} | bf16: {use_bf16}")


## 3. Tokenize abstracts and TLDR targets

BART receives the abstract as input and the human-written TLDR as the target. Padding is handled dynamically by `DataCollatorForSeq2Seq`.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(T["model"])

def preprocess_batch(batch):
    model_inputs = tokenizer(
        batch["abstract"],
        max_length=D["max_source_len"],
        truncation=True
    )
    target_texts = [refs[0] for refs in batch["human_tldrs"]]
    labels = tokenizer(
        text_target=target_texts,
        max_length=D["max_target_len"],
        truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_raw = train_df[["abstract", "human_tldrs"]].to_dict("list")
val_raw = val_df[["abstract", "human_tldrs"]].to_dict("list")

from datasets import Dataset
train_ds = Dataset.from_dict(train_raw).map(
    preprocess_batch, batched=True, remove_columns=["abstract", "human_tldrs"]
)
val_ds = Dataset.from_dict(val_raw).map(
    preprocess_batch, batched=True, remove_columns=["abstract", "human_tldrs"]
)

print("Tokenized examples:", len(train_ds), "train |", len(val_ds), "validation")
print("Example target:", tokenizer.decode(train_ds[0]["labels"], skip_special_tokens=True))


## 4. Configure training and early stopping

Training objective: sequence-to-sequence cross-entropy loss. The model is initialized from the original pretrained `facebook/bart-base`, not from a previously fine-tuned checkpoint.

The best checkpoint is selected by the lowest validation loss. Early stopping ends training after the configured number of non-improving validation epochs.


In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(T["model"])
model.config.use_cache = False
model.gradient_checkpointing_enable()

total_update_steps = math.ceil(len(train_ds) / (per_device_bs * grad_accum)) * int(T["epochs"])
warmup_steps = int(float(T["warmup_ratio"]) * total_update_steps)

# Handle minor argument-name differences across Transformers versions.
arg_params = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
eval_key = "eval_strategy" if "eval_strategy" in arg_params else "evaluation_strategy"

args_kwargs = dict(
    output_dir=str(ROOT / "outputs" / "trainer_checkpoints"),
    num_train_epochs=int(T["epochs"]),
    per_device_train_batch_size=per_device_bs,
    per_device_eval_batch_size=per_device_bs,
    gradient_accumulation_steps=grad_accum,
    learning_rate=float(T["lr"]),
    weight_decay=float(T["weight_decay"]),
    warmup_steps=warmup_steps,
    lr_scheduler_type="linear",
    optim="adamw_torch",
    logging_steps=20,
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=use_fp16,
    bf16=use_bf16,
    gradient_checkpointing=True,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=0,
)
args_kwargs[eval_key] = "epoch"
training_args = Seq2SeqTrainingArguments(**args_kwargs)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=int(T["early_stopping_patience"]))],
)
trainer_params = inspect.signature(Seq2SeqTrainer.__init__).parameters
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)
print("Max epochs:", T["epochs"])
print("Warmup steps:", warmup_steps)
print("Early-stopping patience:", T["early_stopping_patience"])


## 5. Train once and save the best checkpoint

If `outputs/model/` already contains a saved model, the notebook asks you to explicitly choose whether to overwrite it. Do not retrain just to change decoding settings; those are applied later during inference.


In [ ]:
RETRAIN = False  # Change to True only if you intentionally want to train a fresh checkpoint.

if (MODEL_DIR / "config.json").exists() and not RETRAIN:
    print(f"Existing checkpoint found at {MODEL_DIR}; training skipped.")
    print("Set RETRAIN = True only if you intend to replace it.")
else:
    start = time.time()
    train_result = trainer.train()
    trainer.save_model(str(MODEL_DIR))
    tokenizer.save_pretrained(str(MODEL_DIR))

    log_history = trainer.state.log_history
    (OUTPUT_DIR / "train_log.json").write_text(json.dumps(log_history, indent=2))
    summary = {
        "base_model": T["model"],
        "seed": SEED,
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_validation_loss": trainer.state.best_metric,
        "epochs_completed": trainer.state.epoch,
        "per_device_batch_size": per_device_bs,
        "gradient_accumulation_steps": grad_accum,
        "effective_batch_size": per_device_bs * grad_accum,
        "learning_rate": T["lr"],
        "max_epochs": T["epochs"],
        "max_source_len": D["max_source_len"],
        "max_target_len": D["max_target_len"],
        "training_runtime_minutes": round((time.time() - start) / 60, 2),
    }
    (OUTPUT_DIR / "training_summary.json").write_text(json.dumps(summary, indent=2))
    print(json.dumps(summary, indent=2))


## 6. Review training and validation loss

A lower validation loss indicates better fit to the held-out validation targets under teacher-forced loss. It is not, by itself, a complete summary-quality or factuality evaluation.


In [ ]:
log_path = OUTPUT_DIR / "train_log.json"
if log_path.exists():
    history = pd.DataFrame(json.loads(log_path.read_text()))
    train_logs = history.dropna(subset=["loss"]) if "loss" in history else pd.DataFrame()
    eval_logs = history.dropna(subset=["eval_loss"]) if "eval_loss" in history else pd.DataFrame()

    if not train_logs.empty:
        display(train_logs[["epoch", "step", "loss"]].tail(10))
    if not eval_logs.empty:
        display(eval_logs[["epoch", "step", "eval_loss"]])
        best_row = eval_logs.loc[eval_logs["eval_loss"].idxmin()]
        print(f"Lowest validation loss: {best_row['eval_loss']:.4f} at epoch {best_row['epoch']:.2f}")

    plt.figure(figsize=(7, 4))
    if not train_logs.empty:
        plt.plot(train_logs["step"], train_logs["loss"], label="training loss")
    if not eval_logs.empty:
        plt.plot(eval_logs["step"], eval_logs["eval_loss"], marker="o", label="validation loss")
    plt.xlabel("Training step")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No training log yet. Run the training cell first.")


## Deliverables from your training stage

- `outputs/model/`: fine-tuned BART checkpoint and tokenizer files.
- `outputs/train_log.json`: recorded training and validation logs.
- `outputs/training_summary.json`: model name, hyperparameters, best checkpoint, and training runtime.

Your teammates can load `outputs/model/` directly. They do not need to retrain the model to change `num_beams`, sampling settings, or candidate count.
